In [1]:
from google.colab import auth
auth.authenticate_user()


In [2]:
# Instalação dos pacotes
!pip install -q streamlit gspread pyngrok pandas plotly

In [3]:
%%writefile app.py
import streamlit as st
import pandas as pd
import plotly.express as px
import gspread
from google.colab import auth
from google.auth import default

# Autenticação
auth.authenticate_user()
creds, _ = default()
gc = gspread.authorize(creds)

# Carregar dados da planilha
sheet_id = '1mWDAKMT6NvOzltTHPxk2ly1FtufTbPYxn5l_IViKrdM'
worksheet = gc.open_by_key(sheet_id).sheet1
data = worksheet.get_all_records()
df = pd.DataFrame(data)

# Corrigir nomes de colunas e tipos
df.columns = [col.strip() for col in df.columns]
df["Data"] = pd.to_datetime(df["Data"])
df["Quantidade"] = pd.to_numeric(df["Quantidade"], errors='coerce')
df["Valor"] = pd.to_numeric(df["Valor"], errors='coerce')
df["Total"] = pd.to_numeric(df["Total"], errors='coerce')

# Layout
st.set_page_config(layout="wide", page_title="Dashboard de Vendas")
st.title("📊 Dashboard Estratégico de Vendas")

# Filtro por data
datas = df["Data"].sort_values().unique()
data_inicial = st.sidebar.date_input("Data inicial", value=pd.to_datetime(datas.min()))
data_final = st.sidebar.date_input("Data final", value=pd.to_datetime(datas.max()))
df_filtrado = df[(df["Data"] >= pd.to_datetime(data_inicial)) & (df["Data"] <= pd.to_datetime(data_final))]

# Exibir tabela com dados atualizados
st.subheader("📋 Tabela de Vendas")
st.dataframe(df_filtrado, use_container_width=True)

# KPIs
col1, col2, col3, col4 = st.columns(4)
col1.metric("Faturamento Total", f"R$ {df_filtrado['Total'].sum():,.2f}")
col2.metric("Itens Vendidos", int(df_filtrado['Quantidade'].sum()))
col3.metric("Produtos", df_filtrado['Produto'].nunique())
col4.metric("Vendedores", df_filtrado['Vendedor'].nunique())

# Gráfico: Canal de venda
st.subheader("🔹 Total por Canal de Venda")
canal_chart = px.pie(df_filtrado, names="Canal da Venda", values="Total", hole=0.4)
st.plotly_chart(canal_chart, use_container_width=True)

# Gráfico: Forma de pagamento
st.subheader("🔹 Formas de Pagamento")
pagamento_data = df_filtrado.groupby("Forma de pagamento")["Total"].sum().reset_index()
pagamento_chart = px.bar(pagamento_data, x="Forma de pagamento", y="Total", color="Forma de pagamento", title="Faturamento por Forma de Pagamento")
st.plotly_chart(pagamento_chart, use_container_width=True)

# Gráfico: Vendedor
st.subheader("🔹 Ranking de Vendedores")
vendedor_data = df_filtrado.groupby("Vendedor")["Total"].sum().reset_index().sort_values(by="Total", ascending=False)
vendedor_chart = px.bar(vendedor_data, x="Vendedor", y="Total", color="Vendedor", title="Vendas por Vendedor")
st.plotly_chart(vendedor_chart, use_container_width=True)



Overwriting app.py


In [4]:
# ✅ Primeiro configure o token
!ngrok config add-authtoken 2wagHc9ZGZIM9WjzkrplvLoZXGG_6oLCSnmj6BGeuSaXTMN8e


Authtoken saved to configuration file: /root/.config/ngrok/ngrok.yml


In [5]:
# ✅ Depois rode o app e conecte
from pyngrok import ngrok
import time
import threading
import os

def run():
    os.system('streamlit run app.py')

thread = threading.Thread(target=run)
thread.start()
time.sleep(5)

public_url = ngrok.connect("http://localhost:8501")

print(f"🌐 Dashboard disponível em: {public_url}")


🌐 Dashboard disponível em: NgrokTunnel: "https://fe3f-34-82-119-96.ngrok-free.app" -> "http://localhost:8501"
